In [ ]:
# this code is to be run in google colab
# google colab notebook link: https://colab.research.google.com/drive/1pDmPleRUz7o0_q4_FthXtvAwU8dkFrc6#scrollTo=WXgmDyRQlqcw

In [ ]:
# importing the dataset directly from kaggle
import kagglehub

# download latest version
path = kagglehub.dataset_download("vjcalling/speaker-recognition-audio-dataset")

print("Path to dataset files:", path)

In [ ]:
# using Mel-Frequency Cepstral Coefficients (MFCCs) as features

# installing required libraries
!pip install librosa

# loading audio files and extracting MFCC's
import librosa
import numpy as np
import os
import pandas as pd

def extract_mfcc(file_path, n_mfcc=13):
    y, sr = librosa.load(file_path, sr=None)  # Load audio file
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)  # Extract MFCC
    return np.mean(mfcc, axis=1)  # Compute mean across time axis


# path to dataset (UPDATE IF NEEDED!!!)
dataset_path = '/root/.cache/kagglehub/datasets/vjcalling/speaker-recognition-audio-dataset/versions/1' 

In [ ]:
# Check if the path exists
if os.path.exists(dataset_path):
    print("Path exists.")
else:
    print("Path does not exist.")

In [ ]:
# dictionary to store data:
data = []

# iterate through each speaker folder
for speaker in os.listdir(dataset_path):
    speaker_folder = os.path.join(dataset_path, speaker)
    
    # ensuring it's a directory (and NOT a file)
    if os.path.isdir(speaker_folder):
        # interating through each .wav file
        for file in os.listdir(speaker_folder):
            if file.endswith(".wav"):
                file_path = os.path.join(speaker_folder, file)
                
                # MFCC features extract karo
                features = extract_mfcc(file_path)
                
                # dataset mein append karo (speaker label bhi!!!)
                data.append([file_path, speaker] + features.tolist())

In [ ]:
# convert to pandas dataframe
columns = ["file_path", "speaker"] + [f"mfcc_{i}" for i in range(13)]
df = pd.DataFrame(data, columns=columns)

# display dataset header
print(df.head())

# saving extracted features for future use
df.to_csv("audio_features.csv", index=False)
print("Feature extraction completed. Saved as 'audio_features.csv'.")

In [ ]:
# debugging for empty dataframe part 1
# checking contents of dataset folder
import os

dataset_path = "/root/.cache/kagglehub/datasets/vjcalling/speaker-recognition-audio-dataset/versions/1"

for root, dirs, files in os.walk(dataset_path):
    print(f"Directory: {root}")
    print(f"Subdirectories: {dirs}")
    print(f"Files: {files}")
    print("-" * 50)

In [ ]:
# debugging for empty dataframe part 2
# manually checking for .wav files
wav_files = []

for root, _, files in os.walk(dataset_path):
    for file in files:
        if file.endswith(".wav"):
            wav_files.append(os.path.join(root, file))

print(f"Found {len(wav_files)} .wav files")
print(wav_files[:10])  # Print first 10 files to check paths


In [ ]:
# debugging for empty dataframe part 3
# verify that librosa can read an audio file
import librosa

sample_wav = wav_files[0]  # Take the first detected file

try:
    y, sr = librosa.load(sample_wav, sr=None)
    print(f"Loaded {sample_wav} with sample rate {sr}")
except Exception as e:
    print(f"Error loading {sample_wav}: {e}")

In [ ]:
# FEATURE EXTRACTION!
# Path to your dataset
DATASET_PATH = "/root/.cache/kagglehub/datasets/vjcalling/speaker-recognition-audio-dataset/versions/1/50_speakers_audio_data"

# Function to extract audio features
def extract_features(file_path):
    try:
        y, sr = librosa.load(file_path, sr=None)  # Load audio file
        mfccs = np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13), axis=1)  # MFCC (13 Coefficients)
        chroma = np.mean(librosa.feature.chroma_stft(y=y, sr=sr), axis=1)  # Chroma Feature
        spectral_contrast = np.mean(librosa.feature.spectral_contrast(y=y, sr=sr), axis=1)  # Spectral Contrast
        tonnetz = np.mean(librosa.feature.tonnetz(y=librosa.effects.harmonic(y), sr=sr), axis=1)  # Tonnetz Feature
        
        # Concatenating all features into a single array
        features = np.hstack([mfccs, chroma, spectral_contrast, tonnetz])
        return features
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

# List to store extracted features
data = []

# Iterate through speakers' directories
for speaker in os.listdir(DATASET_PATH):
    speaker_path = os.path.join(DATASET_PATH, speaker)
    
    if os.path.isdir(speaker_path):  # Ensure it's a directory
        for file in os.listdir(speaker_path):
            if file.endswith(".wav"):
                file_path = os.path.join(speaker_path, file)
                features = extract_features(file_path)
                
                if features is not None:
                    data.append([speaker] + list(features))  # Append speaker label & features

# Convert to Pandas DataFrame
feature_columns = [f"MFCC_{i}" for i in range(13)] + \
                  [f"Chroma_{i}" for i in range(12)] + \
                  [f"Spectral_{i}" for i in range(7)] + \
                  [f"Tonnetz_{i}" for i in range(6)]

df = pd.DataFrame(data, columns=["Speaker"] + feature_columns)

# Save to CSV for further use
df.to_csv("extracted_features.csv", index=False)

print("Feature extraction completed! Saved as extracted_features.csv")